<a href="https://colab.research.google.com/github/Juliusp36/GinyuForce2/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Step Up

In [1]:
!git clone https://github.com/Juliusp36/GinyuForce2.git
%cd GinyuForce2
!git pull origin main
import sys
sys.path.append("src")

from evaluate import to_matrix
from models import baseline_matrix, rank_k
from evaluate import to_matrix
from models import baseline_matrix, rank_k

Cloning into 'GinyuForce2'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 170 (delta 71), reused 90 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 1.64 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/GinyuForce2
From https://github.com/Juliusp36/GinyuForce2
 * branch            main       -> FETCH_HEAD
Already up to date.


It downloads your GitHub into Colab, pulls the lastest changes, adds the src folder to Python's import path, and then imports to_matrix, baseline_matrix, and rank_k from my files.

In [7]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip
ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t", header=None,
    names=["user", "item", "rating", "timestamp"],
)

replace ml-100k/allbut.pl? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

This downloads the MovieLens 100k dataset, unzips it, and loads the rating file into a pandas table called ratings with columns for user, item, rating, and timestamp.

In [8]:
import pandas as pd
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
header=None, usecols=[1])[1].to_numpy()

print(len(titles), titles[49])

1682 Star Wars (1977)


This loads the movie title from MovieLens file inot an array, then prints how many there are (1682) and the title at index 49.

In [10]:
import numpy as np
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


This builds the full user-by-movie rating matrix, subtracts the baseline predictions and runs an SVD on what's left to find the hidden factors behind people's tastes.

In [12]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f"  {v[m]:+.3f}  {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f"  {v[m]:+.3f}  {titles[m]}")

show_factor(1)

Factor 1, one end:
  -0.087  Clockwork Orange, A (1971)
  -0.079  Trainspotting (1996)
  -0.072  Leaving Las Vegas (1995)
  -0.072  Pulp Fiction (1994)
  -0.069  Brazil (1985)
  -0.067  Fargo (1996)
Factor 1, other end:
  +0.211  Independence Day (ID4) (1996)
  +0.175  Return of the Jedi (1983)
  +0.149  Raiders of the Lost Ark (1981)
  +0.149  Back to the Future (1985)
  +0.136  Braveheart (1995)
  +0.134  Forrest Gump (1994)


It defines a function that picks one hidden factor from the SVD and prints the 6 popular movies with the lowest scores on it, then the 6 with the highest. Those two ends show what taste dimension the factor captures.

In [15]:
show_factor(3)

Factor 3, one end:
  -0.137  Independence Day (ID4) (1996)
  -0.131  Star Trek: First Contact (1996)
  -0.130  Rock, The (1996)
  -0.129  Alien (1979)
  -0.121  Monty Python and the Holy Grail (1974)
  -0.119  Aliens (1986)
Factor 3, other end:
  +0.163  Sound of Music, The (1965)
  +0.144  Grease (1978)
  +0.099  E.T. the Extra-Terrestrial (1982)
  +0.087  Mary Poppins (1964)
  +0.085  Jaws 2 (1978)
  +0.084  Snow White and the Seven Dwarfs (1937)


In [14]:
def find(word):
    """Print the ID and title of every movie whose title contains word."""
    for m, t in enumerate(titles):
        if word.lower() in t.lower():
            print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


It defines a function that searches all the movie titles for a word and prints the ID and title of each match.

In [16]:
my_ratings = {50: 5, 172: 5, 183: 5, 195: 5, 89: 4, 1: 4,

143: 1, 451: 1, 88: 1, 739: 2}

In [18]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)
known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


It adds you as a new user: it makes a blank row, fills in your rating from my_rating, and attaches it as an extra row as the bottom of the rating matrix. It then redoes the baseline and SVD with you included, and builds P, a matrix of predicted ratings for every user and movie using the top 10 hidden factors plus the baseline.

In [20]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    """The n highest-scoring movies among those we may recommend."""
    return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
    print(f"  {mine[m]:.2f}  {titles[m]}")

print("Baseline recommends:")
for m in top(base):
    print(f"  {base[m]:.2f}  {titles[m]}")

print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
  4.28  Close Shave, A (1995)
  4.26  Wrong Trousers, The (1993)
  4.20  Fargo (1996)
  4.15  Casablanca (1942)
  4.14  Raiders of the Lost Ark (1981)
  4.11  Wallace & Gromit: The Best of Aardman Animation (1996)
  4.08  Godfather, The (1972)
  4.07  Schindler's List (1993)
  4.07  Shawshank Redemption, The (1994)
  4.06  Return of the Jedi (1983)
Baseline recommends:
  4.23  Close Shave, A (1995)
  4.15  Wrong Trousers, The (1993)
  4.13  Schindler's List (1993)
  4.13  Shawshank Redemption, The (1994)
  4.09  Casablanca (1942)
  4.08  Wallace & Gromit: The Best of Aardman Animation (1996)
  4.08  Usual Suspects, The (1995)
  4.06  Good Will Hunting (1997)
  4.01  Titanic (1997)
  4.01  12 Angry Men (1957)
in both lists: 6


It prints two top 10 recommendation lists for you and compares them. mine is the row of predictions from the SVD model, and base is the plain baseline prediction.

In [21]:
personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
 +0.34 Fargo (1996)
 +0.34 Return of the Jedi (1983)
 +0.25 Star Trek: The Wrath of Khan (1982)
 +0.22 Star Trek: First Contact (1996)
 +0.21 2001: A Space Odyssey (1968)
 +0.20 Raiders of the Lost Ark (1981)
Moved down most for me:
 -0.36 English Patient, The (1996)
 -0.21 Liar Liar (1997)
 -0.19 Dirty Dancing (1987)
 -0.19 Mr. Holland's Opus (1995)
 -0.18 Ghost (1990)
 -0.17 Mrs. Doubtfire (1993)


It shows what the hidden-factor model changed for me specifically. personal = mine - base is the gap between the SVD prediction and the plain baseline prediction for each movie, so it isolates the part the comes from the personal taste rather than general popularity.